# Validation of the $\beta_i$ index rule

## Cross-class priority rule

In the Bassamboo framework, servers should be allocated to the class with the highest index

$
\beta_i = p_i \mu_i
$

where

$
p_i
$

is the abandonment penalty for class $i$ and

$
\mu_i
$

is the service rate.

When service rates are identical across classes, the rule reduces to

$
\beta_i \propto p_i.
$

So the class with the larger abandonment penalty should receive priority.

## Policies tested

We compare two simple priority rules:

**priority_0**

$$
\text{class 0} \rightarrow \text{class 1}
$$

**priority_1**

$$
\text{class 1} \rightarrow \text{class 0}
$$

If the $$\beta_i$$ rule is correct, the second policy should produce a substantially lower weighted abandonment cost.

## Performance metric

The objective is the weighted abandonment cost

$$
\text{Cost} = p_0 A_0 + p_1 A_1
$$

where

$$
A_i
$$

is the number of abandonments in class $$i$$.

## Interpreting the results

The results show that prioritizing class 1 dramatically reduces the weighted abandonment cost.

Importantly:

- the **total number of abandonments remains similar**
- the improvement comes from **reallocating service toward the higher-penalty class**

This is exactly the qualitative behavior predicted by the $$\beta_i$$ index rule.

In [5]:
import pandas as pd
import numpy as np

from gi_gi_n_gi_multiclass.core.engine_multi import run_sim_multi
from gi_gi_n_gi_multiclass.core.types_multi import CustomerClass, MultiSimConfig
from gi_gi_n_gi_multiclass.dists.common import Exponential
from gi_gi_n_gi_multiclass.outputs.schemas import customers_to_dataframe
from gi_gi_n_gi_multiclass.policies.priority_fcfs import PriorityFCFS

In [6]:
N = 10

class0 = CustomerClass(
    name="class0",
    interarrival=Exponential(rate=8),
    service=Exponential(rate=1),
    patience=Exponential(rate=0.5),
)

class1 = CustomerClass(
    name="class1",
    interarrival=Exponential(rate=12),
    service=Exponential(rate=1),
    patience=Exponential(rate=0.5),
)

classes=[class0,class1]

In [7]:
p0 = 1
p1 = 5

def run_policy(policy):

    cfg = MultiSimConfig(
        n_servers=N,
        classes=classes,
        policy=policy,
        seed=0,
        warmup_time=500,
        run_time=5000,
    )

    res = run_sim_multi(cfg)

    df = customers_to_dataframe(res)

    aband0 = ((df.class_id==0) & (df.outcome=="ABANDONED")).sum()
    aband1 = ((df.class_id==1) & (df.outcome=="ABANDONED")).sum()

    cost = p0*aband0 + p1*aband1

    return {
        "aband0":aband0,
        "aband1":aband1,
        "cost":cost
    }

In [8]:
results = []

results.append(
    {"policy":"priority_0", **run_policy(PriorityFCFS([0,1]))}
)

results.append(
    {"policy":"priority_1", **run_policy(PriorityFCFS([1,0]))}
)

pd.DataFrame(results)

,policy,aband0,aband1,cost
0,priority_0,5390,50031,255545
1,priority_1,40108,15174,115978


## Takeaway

This experiment confirms that the simulator correctly reproduces the key intuition behind the Bassamboo allocation rule.

When service rates are equal, prioritizing the class with the larger penalty

$$
p_i
$$

reduces the weighted abandonment cost.

This validates the cross-class component of the mTIQ policy.